# 04 — Recalibration, firing controls, and G-POS

The repaired swap, independent direction gate, and READ validation have all
completed before this notebook runs. Stage 2 now re-verifies G-SWAP, executes
the redesigned controls whose measured metrics actually contain the
suppressed token, and tests the known-narration positive control. Every result
is persisted before the workflow chooses either Stage-3 science or the
Stage-4 replication-failure report.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['HF_HOME'] = str(Path.home() / '.cache/huggingface')
os.environ['HF_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')

metrics = json.loads((ROOT / 'results/metrics.json').read_text())
repair = metrics['repair_v2']
ledger = repair['gate_ledger']
assert ledger['g_swap'] == 'PASS'
assert ledger['g_dir'] in {'PASS', 'DROPPED_MD'}
assert ledger['read_validation'] == 'PASS'
assert ledger['stage3_science'] == 'PROHIBITED'
workspace_layers = repair['stage1']['g_swap']['canonical_configuration']['layers']

from src.jlens_iface import load_published_lens
from src.model_utils import load_model
from src.v2_repair import MODEL_ID

bundle = load_model(MODEL_ID)
lens = load_published_lens(MODEL_ID)
print({
    'model': bundle.model_id,
    'workspace_layers': workspace_layers,
    'g_swap': ledger['g_swap'],
    'g_dir': ledger['g_dir'],
    'read_validation': ledger['read_validation'],
    'science_before_recalibration': ledger['stage3_science'],
})

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

{'model': 'Qwen/Qwen2.5-7B-Instruct', 'workspace_layers': [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24], 'g_swap': 'PASS', 'g_dir': 'PASS', 'read_validation': 'PASS', 'science_before_recalibration': 'PROHIBITED'}


## Execute every Stage-2 calibration arm

Failure is recorded rather than raised here. In particular, the notebook
prints the direct and language suppression effects so a nominal control PASS
cannot hide another structural-zero design.

In [2]:
from src.v2_recalibration import run_stage2

stage2 = run_stage2(bundle, lens, workspace_layers=workspace_layers)
criteria = stage2.get('criteria', {
    'g_swap_reverified': stage2['g_swap_reverification']['status'] == 'PASS',
    'controls_fire': stage2['controls_fire']['status'] == 'PASS',
    'matched_random_specificity': stage2['random_pair_null']['status'] == 'PASS',
    'absent_coordinate_specificity': stage2['absent_coordinate_null']['status'] == 'PASS',
    'capability_preserved': stage2['capability']['status'] == 'PASS',
    'g_pos_reproduced': stage2['g_pos']['status'] == 'PASS',
})
print(json.dumps({
    'stage2': stage2['status'],
    'criteria': criteria,
    'identity_j_baseline': stage2['identity_j_baseline']['status'],
    'stage3_allowed': stage2['stage3_allowed'],
    'stage4_required': stage2['stage4_required'],
}, indent=2))

print('\nDirect concept-answer controls:')
for row in stage2['controls_fire']['direct_concept_probes']['rows']:
    print({
        'item': row['name'],
        'clean_pairwise_correct': row['clean_pairwise_correct'],
        'internal_delta': row['internal_delta'],
        'suppression_delta': row['suppression_delta'],
        'suppression_fired': row['suppression_fired'],
    })

print('\nLanguage controls with firing metrics:')
for row in stage2['controls_fire']['language_controls']['rows']:
    print(row)

print('\nKnown-narration passage decisions:')
for row in stage2['g_pos']['rows']:
    print({
        'key': row['key'],
        'category': row['category'],
        'minimum_write_rank': row['minimum_all_prompt_language_rank'],
        'instruction_rank_diagnostic': row['minimum_instruction_span_language_rank_diagnostic'],
        'automatic_internal_delta': row['automatic_internal_delta'],
        'primary_weight_read_ratio': row['primary_weight_read_ratio_auto_over_direct'],
        'attribution_read_ratio_secondary': row['attribution_read_ratio_auto_over_direct_secondary'],
        'failed_checks': [
            name for name, passed in row['checks'].items() if not passed
        ],
        'joint_reproduction': row['joint_reproduction'],
    })

{
  "stage2": "FAIL",
  "criteria": {
    "g_swap_reverified": true,
    "controls_fire": true,
    "random_pair_specific": true,
    "absent_coordinate_specific": true,
    "capability_preserved": false,
    "g_pos_reproduced": false
  },
  "identity_j_baseline": "COMPUTED_DIAGNOSTIC",
  "stage3_allowed": false,
  "stage4_required": true
}

Direct concept-answer controls:
{'item': 'spider-legs', 'clean_pairwise_correct': True, 'internal_delta': -33.578125, 'suppression_delta': -1.0, 'suppression_fired': True}
{'item': 'animal-legs-buffalo2', 'clean_pairwise_correct': True, 'internal_delta': -61.0625, 'suppression_delta': -1.0, 'suppression_fired': True}
{'item': 'chem-photosynthesis-Z', 'clean_pairwise_correct': True, 'internal_delta': -41.375, 'suppression_delta': -1.0, 'suppression_fired': True}

Language controls with firing metrics:
{'key': 'fr1', 'category': 'French', 'source_effect': -0.9999998807907104, 'english_effect': 1.0, 'direct_label_effect': -1.0}
{'key': 'fr2', 'categor

## Persist first, then branch

A failed calibration arm is itself the gate result. It sends the workflow to
notebook 08 without licensing P1–P3; it does not become an exception that
prevents the evidence from being saved.

In [3]:
from src.v2_recalibration import persist_stage2

metrics = persist_stage2(stage2)
ledger = metrics['repair_v2']['gate_ledger']
expected_branch = (
    'ALLOWED' if stage2['stage3_allowed'] else 'SKIPPED_PREREQUISITE'
)
assert stage2['status'] == ('PASS' if stage2['stage3_allowed'] else 'FAIL')
assert ledger['g_swap_reverify'] == stage2['g_swap_reverification']['status']
assert ledger['controls_fire'] == stage2['controls_fire']['status']
assert ledger['random_pair_null'] == stage2['random_pair_null']['status']
assert ledger['absent_coordinate_null'] == stage2['absent_coordinate_null']['status']
assert ledger['capability'] == stage2['capability']['status']
assert ledger['g_pos'] == stage2['g_pos']['status']
assert ledger['stage3_science'] == expected_branch
next_notebook = (
    '05_science_twohop'
    if expected_branch == 'ALLOWED'
    else '05-07_skip_guards_then_08_report_stage4'
)
print(json.dumps({
    'persisted_stage2': stage2['status'],
    'gate_ledger': ledger,
    'next': next_notebook,
    'hypothesis_inference_allowed': expected_branch == 'ALLOWED',
}, indent=2))

{
  "persisted_stage2": "FAIL",
  "gate_ledger": {
    "controls_fire": "PASS",
    "g1": "PASS",
    "g_dir": "PASS",
    "g_pos": "FAIL",
    "g_swap": "PASS",
    "read_validation": "PASS",
    "stage0_preflight": "PASS",
    "stage3_science": "SKIPPED_PREREQUISITE",
    "upstream_causal_swap": "NOT_RUNNABLE_RELEASE_OMISSION",
    "upstream_readout": "PASS",
    "g_swap_reverify": "PASS",
    "random_pair_null": "PASS",
    "absent_coordinate_null": "PASS",
    "capability": "FAIL",
    "stage4_report": "REQUIRED"
  },
  "next": "05-07_skip_guards_then_08_report_stage4",
  "hypothesis_inference_allowed": false
}


In [4]:
import gc
import torch

del stage2, metrics, lens, bundle
gc.collect()
torch.cuda.empty_cache()
print(
    'Notebook 04 complete. Continue to Stage 3 only when the persisted '
    'ledger says ALLOWED; otherwise execute the Stage-4 report.'
)

Notebook 04 complete. Continue to Stage 3 only when the persisted ledger says ALLOWED; otherwise execute the Stage-4 report.
